# R3_05 — Panggilan LLM (arbitrase, LLM-always, LLM-only)

* Fungsi `build_prompt` dan `call_llm` **asli NB06** (salinan verbatim di `legacy_code/`), snapshot `C.LLM_MODEL`
  (`qwen3.7-max-2026-06-08`; pengganti `qwen3-max-2026-01-23` yang ditarik penyedia — lihat `ganti_model_llm.md`),
  temperature 0, `max_tokens` 256, thinking off, maks. 3 percobaan, fallback sama dengan NB06. Prompt LLM-only = prompt baseline
  `revision/04` (salinan verbatim). API key **hanya** dari environment variable `DASHSCOPE_API_KEY`.
* **Arbitrase:** kasus yang ditandai R3_04. **LLM-always:** semua pasangan dengan prompt arbitrase. **LLM-only:** semua pasangan dengan
  prompt baseline. Ketiganya dipanggil dengan model yang sama; respons qwen3-max lama **tidak** dipakai (kunci cache = hash yang memuat
  nama model, dan cache lama tidak dibaca untuk kondisi utama).
* Cache baru: `outputs_round3/llm/llm_cache/responses.jsonl` (respons, token, latensi). Bila panggilan berhenti karena galat API
  (kuota/saldo, autentikasi, model, koneksi), notebook **berhenti tanpa fallback** dan dapat dijalankan ulang: respons yang sudah ada
  dipakai dari cache. Fallback NB06 hanya untuk respons yang tidak dapat di-parse.
* Analisis sensitivitas (mode nyata): LLM-only `qwen3-max-2026-01-23` dari cache arsip (`C.LLM_CACHE_ARCHIVE`, tanpa API) →
  `llm_only_qwen3max_test.csv`; dibandingkan di R3_06. Tidak dicampur ke kondisi utama.
* **`RUN_API = False` (default):** hanya estimasi jumlah panggilan/token/biaya dan pemakaian cache; tidak ada panggilan API.
* **`DRY_RUN = True` (default):** memakai keluaran R3_04 mode dry (data dev). Dengan `DRY_MOCK_LLM = True`, permintaan yang tidak
  ada di cache diisi label buatan (ditandai `mock`) agar R3_06/R3_07 dapat diuji tanpa API. Mode nyata tidak pernah memakai mock.

In [ ]:
# ── PARAMETER ──
DRY_RUN = True
RUN_API = False                 # True = boleh memanggil API untuk permintaan yang belum ada di cache
DRY_MOCK_LLM = True             # hanya berlaku bila DRY_RUN=True
MAX_NEW_CALLS = None            # batas pengaman jumlah panggilan API baru (None = tanpa batas)

import os
DRY_RUN = os.environ.get('R3_DRY_RUN', str(DRY_RUN)) == 'True'
RUN_API = os.environ.get('R3_RUN_API', str(RUN_API)) == 'True'

In [ ]:
import sys, json, time, random
sys.path.insert(0, os.environ.get('R3_CODE_DIR', os.getcwd()))
import numpy as np, pandas as pd
import config as C
import r3_lib as L
D = L.mode_dirs(DRY_RUN)
if DRY_RUN:
    L.forbid_test_access(allow_dirs=[C.DRY_DIR])
frozen = L.verify_frozen(strict=not DRY_RUN, check_models=False)
IN, OUT = D.test_pred, C.assert_writable(D.llm)
assert (IN / 'DONE.json').exists(), f'R3_04 ({"dry" if DRY_RUN else "nyata"}) belum selesai: {IN}'
OUT.mkdir(parents=True, exist_ok=True)
NEW_CACHE, ATTEMPTS = OUT / 'llm_cache' / 'responses.jsonl', OUT / 'llm_cache' / 'attempts.jsonl'
print(f"Mode: {'DRY_RUN (dev)' if DRY_RUN else 'NYATA'} | RUN_API={RUN_API} | output: {OUT}")

In [ ]:
ens = pd.read_csv(IN / 'ensemble_test.csv'); ens['pair_id'] = ens['pair_id'].astype(str)
flags = pd.read_csv(IN / 'flags.csv'); flags['pair_id'] = flags['pair_id'].astype(str)
pairs = pd.read_csv(D.pairs, usecols=['pair_id', 'sentence_a'])          # tanpa label
pairs['pair_id'] = pairs['pair_id'].astype(str)
df = ens.merge(pairs, on='pair_id').merge(flags[['pair_id', 'needs_llm']], on='pair_id')
assert len(df) == len(ens)
nb6 = L.exec_legacy(['nb06_prompt', 'nb06_call_llm', 'rev04_baseline_prompt'],
                    L.base_namespace(json=json, time=time, random=random))
spec = L.nb06_call_spec()
inv = C.LABEL_MAP_INV

def arbitration_prompt(r):        # sama dengan loop NB6 / revision/04
    return nb6['build_prompt'](str(r['sentence_a']), r['aspek'], inv.get(int(r['soft_vote_label']), 'tidak_disebutkan'),
                               inv.get(int(r['pred_indobert']), 'tidak_disebutkan'), inv.get(int(r['pred_xlm_roberta']), 'tidak_disebutkan'),
                               inv.get(int(r['pred_mbert']), 'tidak_disebutkan'))

def key_of(prompt, model=C.LLM_MODEL):
    return L.llm_cache_key(model, [{'role': 'system', 'content': spec['system_prompt']}, {'role': 'user', 'content': prompt}],
                           spec['temperature'], spec['max_tokens'], spec['extra_body'])

df['prompt_arb'] = df.apply(arbitration_prompt, axis=1)
df['prompt_only'] = df.apply(lambda r: nb6['baseline_prompt'](str(r['sentence_a']), r['aspek']), axis=1)
print(f"{len(df)} pasangan; dirujuk ke LLM: {int(df.needs_llm.sum())}")

## Estimasi SEBELUM memanggil API

In [ ]:
from types import SimpleNamespace
class LiveWithError:
    """Meneruskan ke klien OpenAI dan mencatat galat API terakhir (galat API ≠ respons tak ter-parse; lihat run_one)."""
    def __init__(self, c):
        self.c, self.last_error = c, None
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
    def _create(self, **kw):
        try:
            return self.c.chat.completions.create(**kw)
        except Exception as e:
            self.last_error = e
            raise

live = None
if RUN_API:
    if not os.environ.get(C.LLM_API_KEY_ENV):
        raise RuntimeError(f'{C.LLM_API_KEY_ENV} tidak ada di environment')
    from openai import OpenAI
    live = LiveWithError(OpenAI(api_key=os.environ[C.LLM_API_KEY_ENV], base_url=L.nb06_base_url()))
# Kondisi utama: hanya cache baru (model C.LLM_MODEL). Cache qwen3-max lama TIDAK dibaca → tidak tercampur.
client = L.CachedLLMClient([], NEW_CACHE, ATTEMPTS, live)

# Cache arsip (data UJI, qwen3-max) hanya untuk statistik token estimasi dan analisis sensitivitas; tidak dibaca di DRY_RUN
OLD_CACHES = [] if DRY_RUN else [C.LLM_CACHE_ARCHIVE]
old = [json.loads(l) for p in OLD_CACHES if p.exists() for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]
oldf = pd.DataFrame(old)
if len(oldf):
    oldf['run'] = oldf.tag.str.split(':').str[0]
ref = {r: oldf[oldf.run == t] if len(oldf) else pd.DataFrame() for r, t in
       [('arbitrase', 'pinned_arbitration'), ('LLM-always', 'pinned_llm_always'), ('LLM-only', 'pinned_llm_only')]}
plan = {'arbitrase': df[df.needs_llm].prompt_arb, 'LLM-always': df.prompt_arb, 'LLM-only': df.prompt_only}
# Token/latensi estimasi diambil dari respons qwen3-max lama (prompt sama; tokenizer bisa sedikit berbeda) — hanya perkiraan
rows = []
for name, prompts in plan.items():
    keys = prompts.map(key_of)
    cached = keys.map(client.has)
    uniq_new = keys[~cached].nunique()
    r0 = ref[name] if len(ref[name]) else (oldf if len(oldf) else pd.DataFrame({'prompt_tokens': [340], 'completion_tokens': [57], 'latency_s': [1.4]}))
    pin, pout, lat = r0.prompt_tokens.mean(), r0.completion_tokens.mean(), r0.latency_s.median()
    rows.append({'run': name, 'permintaan': len(prompts), 'ada_di_cache': int(cached.sum()), 'panggilan_baru_unik': int(uniq_new),
                 'token_input_est': round(uniq_new * pin), 'token_output_est': round(uniq_new * pout),
                 'biaya_USD_est': round((uniq_new * pin * C.PRICE_INPUT_PER_1M + uniq_new * pout * C.PRICE_OUTPUT_PER_1M) / 1e6, 4),
                 'waktu_menit_est': round(uniq_new * (lat + 0.3) / 60, 1)})
estimate = pd.DataFrame(rows)
# Arbitrase dan LLM-always berbagi prompt untuk kasus yang ditandai → panggilan unik total:
all_new = pd.concat([plan['arbitrase'], plan['LLM-always'], plan['LLM-only']]).map(key_of)
n_new_total = int(all_new[~all_new.map(client.has)].nunique())
estimate.to_csv(OUT / 'estimate_before_calls.csv', index=False, encoding='utf-8-sig')
print(f'Model: {C.LLM_MODEL} | panggilan API baru unik (arbitrase + LLM-always + LLM-only): {n_new_total}')
estimate

## Panggilan (hanya bila RUN_API=True untuk permintaan yang belum ada di cache)

In [ ]:
class ApiStopped(RuntimeError):
    pass

def run_one(prompt, tag, cl, model=C.LLM_MODEL):
    k = key_of(prompt, model)
    if cl.live is None and not cl.has(k):
        return None, 'not_run'
    cl.counter = {}; cl.current_tag = tag
    was_cached = cl.has(k)
    if cl.live is not None:
        cl.live.last_error = None
    res = nb6['call_llm'](prompt, 'alibaba', cl, model, C.LLM_MAX_RETRIES, C.LLM_RETRY_DELAY_SEC)
    if res is None and cl.live is not None and cl.live.last_error is not None:
        # Galat API (kuota/saldo, autentikasi, model, koneksi) setelah semua percobaan → berhenti TANPA fallback.
        # Respons yang sudah diterima tersimpan di cache baru; jalankan ulang untuk melanjutkan dengan model yang sama.
        raise ApiStopped(f'{tag}: galat API — {cl.live.last_error!r}'[:600])
    if not was_cached:
        time.sleep(0.3)                              # jeda antar-request seperti loop NB6
    return res, ('ok' if res is not None else 'failed')

new_calls = 0
def run_batch(sub, col, run_name, cl, fallback, mock_fn, model=C.LLM_MODEL):
    global new_calls
    out = []
    for _, r in sub.iterrows():
        k = key_of(r[col], model)
        if MAX_NEW_CALLS is not None and not cl.has(k) and cl.live is not None and new_calls >= MAX_NEW_CALLS:
            res, st = None, 'not_run'
        else:
            if not cl.has(k) and cl.live is not None:
                new_calls += 1
            res, st = run_one(r[col], f'{run_name}:{r.pair_id}', cl, model)
        mock = False
        if st == 'failed':
            res = fallback(r)
        elif st == 'not_run' and DRY_RUN and DRY_MOCK_LLM:
            res, mock = mock_fn(r), True
        out.append({'pair_id': r.pair_id, 'status': st, 'mock': mock, 'cache_source': cl.source.get(k),
                    **({'decision': res['decision'], 'final_label': res['final_label'], 'reasoning': res['reasoning']} if res else
                       {'decision': None, 'final_label': None, 'reasoning': None})})
    return pd.DataFrame(out)

fb_arb = lambda r: {'decision': 'retain', 'final_label': inv[int(r['soft_vote_label'])], 'reasoning': 'LLM call failed — fallback to soft vote'}
fb_only = lambda r: {'decision': 'retain', 'final_label': 'tidak_disebutkan', 'reasoning': 'LLM call failed'}
# MOCK (hanya DRY_RUN): label buatan yang bervariasi agar jalur retain/revise di R3_06/R3_07 teruji — BUKAN keluaran LLM
def mock_arb(r):
    ib, sv = int(r['pred_indobert']), int(r['soft_vote_label'])
    return {'decision': 'revise' if ib != sv else 'retain', 'final_label': inv[ib], 'reasoning': 'MOCK (DRY_RUN)'}
mock_only = lambda r: {'decision': 'retain', 'final_label': inv[int(r['pred_mbert'])], 'reasoning': 'MOCK (DRY_RUN)'}
mock_sens = lambda r: {'decision': 'retain', 'final_label': inv[int(r['pred_xlm_roberta'])], 'reasoning': 'MOCK (DRY_RUN)'}
try:
    arb = run_batch(df[df.needs_llm], 'prompt_arb', 'r3_arbitration', client, fb_arb, mock_arb)
    always = run_batch(df, 'prompt_arb', 'r3_llm_always', client, fb_arb, mock_arb)
    only = run_batch(df, 'prompt_only', 'r3_llm_only', client, fb_only, mock_only)
except ApiStopped as e:
    raise RuntimeError(f'BERHENTI: {e}\nPanggilan API baru sesi ini: {new_calls}. Tidak ada label yang ditulis. Perbaiki penyebabnya '
                       f'(mis. kuota/saldo), lalu jalankan ulang notebook ini — respons yang sudah ada dipakai dari cache '
                       f'({NEW_CACHE}). Model TIDAK boleh diganti.') from None
# Sensitivitas: LLM-only qwen3-max dari cache arsip (tanpa API; prosedur & parsing sama). Tidak dipakai di kondisi utama.
SENS_ATTEMPTS = OUT / 'llm_cache' / 'attempts_sensitivitas.jsonl'
if DRY_RUN:     # cache arsip berisi data uji → tidak dibaca; label mock agar jalur R3_06 teruji
    sens = run_batch(df, 'prompt_only', 'sens_llm_only_qwen3max', L.CachedLLMClient([], OUT / 'llm_cache' / 'unused_sens.jsonl', SENS_ATTEMPTS, None),
                     fb_only, mock_sens, model=C.LLM_MODEL_OLD)
else:
    sens = run_batch(df, 'prompt_only', 'sens_llm_only_qwen3max', L.CachedLLMClient([], C.LLM_CACHE_ARCHIVE, SENS_ATTEMPTS, None),
                     fb_only, mock_sens, model=C.LLM_MODEL_OLD)
    assert (sens.status != 'not_run').all(), f"{int((sens.status == 'not_run').sum())} prompt LLM-only tidak ada di cache arsip qwen3-max"
for name, t in [('arbitrase', arb), ('LLM-always', always), ('LLM-only', only), ('sens. qwen3-max', sens)]:
    print(f"{name:15s}: {t.status.value_counts().to_dict()} | mock: {int(t.mock.sum())}")

## Simpan label akhir (format sama dengan final_labels_test.csv NB6)

In [ ]:
complete = all((t.status != 'not_run').all() or (DRY_RUN and DRY_MOCK_LLM) for t in (arb, always, only))
a_map = arb.set_index('pair_id')
rows = []
for _, r in df.iterrows():
    soft = inv[int(r.soft_vote_label)]
    if r.needs_llm and r.pair_id in a_map.index and pd.notna(a_map.loc[r.pair_id, 'final_label']):
        x = a_map.loc[r.pair_id]; decision, final, reason, st = x.decision, x.final_label, x.reasoning, x.status
    elif r.needs_llm:
        decision, final, reason, st = None, None, None, 'not_run'
    else:
        decision, final, reason, st = 'auto_accept', soft, '', 'auto_accept'
    rows.append({'pair_id': r.pair_id, 'aspek': r.aspek, 'soft_vote_label': soft, 'decision': decision, 'final_label': final,
                 'final_label_int': C.LABEL_MAP.get(final, 0) if final else None, 'reasoning': reason, 'status': st,
                 'llm_model': C.LLM_MODEL if r.needs_llm else ''})
final = pd.DataFrame(rows)
for name, t in [('final_labels_test.csv', final), ('llm_always_test.csv', always), ('llm_only_test.csv', only),
                ('llm_only_qwen3max_test.csv', sens)]:
    if 'final_label' in t and name != 'final_labels_test.csv':
        t = t.assign(final_label_int=t.final_label.map(C.LABEL_MAP))
    t.to_csv(OUT / name, index=False, encoding='utf-8-sig')
print(final.decision.value_counts(dropna=False).to_dict())

## Panggilan, token, waktu API (hanya panggilan sungguhan, dari attempts.jsonl)

In [ ]:
att = pd.read_json(ATTEMPTS, lines=True) if ATTEMPTS.exists() and ATTEMPTS.stat().st_size else pd.DataFrame()
usage = pd.DataFrame()
if len(att) and (~att.from_cache).any():
    a = att[(~att.from_cache) & att.ok].copy(); a['run'] = a.tag.str.split(':').str[0]
    usage = a.groupby('run').agg(panggilan=('key', 'size'), token_input=('prompt_tokens', 'sum'), token_output=('completion_tokens', 'sum'),
                                 latensi_median_s=('latency_s', 'median'), latensi_p95_s=('latency_s', lambda x: x.quantile(.95)),
                                 waktu_total_s=('latency_s', 'sum')).reset_index()
    usage['biaya_USD'] = (usage.token_input * C.PRICE_INPUT_PER_1M + usage.token_output * C.PRICE_OUTPUT_PER_1M) / 1e6
    usage['gagal'] = usage.run.map(att[(~att.from_cache) & ~att.ok].assign(run=lambda d: d.tag.str.split(':').str[0]).groupby('run').size()).fillna(0)
usage.to_csv(OUT / 'usage_summary.csv', index=False, encoding='utf-8-sig')
print('Panggilan API baru sesi ini:', new_calls)
display(usage)
# Satu model untuk semua kondisi utama: setiap respons di cache baru harus dari C.LLM_MODEL
new_recs = [json.loads(l) for l in NEW_CACHE.read_text(encoding='utf-8').splitlines() if l.strip()] if NEW_CACHE.exists() else []
models_seen = sorted({(r['model'], r['response_model']) for r in new_recs})
print('Model di cache baru (diminta, dijawab):', models_seen)
assert all(m == C.LLM_MODEL and rm == C.LLM_MODEL for m, rm in models_seen), 'cache baru memuat model lain — berhenti'
if complete:
    L.write_json(OUT / 'DONE.json', {'finished_at': L.now_utc(), 'dry_run': DRY_RUN, 'llm_model': C.LLM_MODEL,
                                     'sensitivity_model': C.LLM_MODEL_OLD, 'models_in_new_cache': models_seen,
                                     'mock_used': bool(arb.mock.any() or always.mock.any() or only.mock.any()),
                                     'counts': {n: t.status.value_counts().to_dict() for n, t in [('arbitrase', arb), ('LLM-always', always), ('LLM-only', only)]},
                                     'decisions': final.decision.value_counts(dropna=False).to_dict(),
                                     'outputs_sha256': {p.name: L.sha256_file(p) for p in sorted(OUT.glob('*.csv'))}})
    print('✓ Lengkap — R3_06 dapat dijalankan')
else:
    print('Belum lengkap (ada permintaan not_run). Set RUN_API=True untuk memanggil API, lalu jalankan ulang (cache dilanjutkan).')